# MAP Scoring using the Bag-of-Words (BoW) Approach


This file serves to create the MAP scores based on the BoW approach, i.e. the MAP Dictionary developed by [Qiu et al. (2023)](https://www.sciencedirect.com/science/article/pii/S1044500522000361)., using 10-K filings of S&P 500 firms with filing year 2013 to 2023.

<div class='alert-warning'>
Libraries
</div>
First, we Import all necessary libraries. These inlcude 'os' to set and handle working directories, 'pandas' and 'numpy' for data handling and calculations, 'pickle' to load and save the prepared data as memory efficient pickle files, 'spacy' for NLP tasks, 'lseg.data' to retrieve further information from the Refinitiv Datastram API, and 'plotnine' to create plots/figures. 

In [1]:
import os
import pandas as pd
import numpy as np
import re
import time
import pickle
import spacy
#import lseg.data as rd # just import this if you want to use the lseg.data package for loading the industry classification data
import plotnine
from scipy.stats.mstats import winsorize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
# package for plot scales
from mizani.formatters import comma_format # (thousands seperator format)

<div class='alert-warning'>
Set the working directory
</div>

In [ ]:
# Set working directory 
os.chdir('../../../data')

<div class='alert-warning'>
Load the pre-processed corpus dataframe & join industry information 
</div>

We first load the file which will be used for the GLLM approach 'GLLM/Corpus_df_GLLM_dimension_scores_final.pkl' to extract the filing keys and ensure that all approaches cover the same firm_year observations. Afterwards, we load the 'Corpus_df_HTML_cleaned_BoW_final.pkl' corpus and filter the observations for the extracted keys. 

NOTE: You need to run the notebook 'MAP_Scoring_GLLM_Approach_final.ipynb' before, to make sure that the file 'Corpus_df_GLLM_dimension_scores_final.pkl' includes the industry information. Alternatively, you can retrieve them with 'rd.get_data(universe, ['TR.CIKNUMBER', 'TR.CompanyName', 'TR.NAICSSector', 'TR.NAICSSubsector', 'TR.NAICSIndustryGroup'])' from the LSEG API. See the mentioned notebook for more details.

In [ ]:
# Load GLLM cleaned corpus to extract filing keys
with open('GLLM/Corpus_df_GLLM_dimension_scores_final.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

corpus_df_keys = corpus_df[['filing_key', 'NAICS Sector Name',
       'NAICS Subsector Name', 'NAICS Subsector All Code']].copy()

del corpus_df

# Load BoW cleaned corpus
with open('BoW/Intermediate_datasets/Corpus_df_HTML_cleaned_BoW_final.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

# Filter BoW corpus for keys present in GLLM corpus
corpus_df = corpus_df[corpus_df['filing_key'].isin(corpus_df_keys['filing_key'])].reset_index(drop=True)

# Merge industry information from GLLM corpus
corpus_df = pd.merge(corpus_df, corpus_df_keys, on='filing_key', how='left')

# Delete duplicates if any exist in the BoW corpus based on filing_key and reset the index
corpus_df = corpus_df.drop_duplicates(subset=['filing_key']).reset_index(drop=True)

del corpus_df_keys

## MAP Dimension Scoring

<div class='alert-info'>
Step 1: Count MAP-related (compound) tokens based on the Qiu et al. (2023) dictionary
</div>

First, we need to import the dictionary from Qiu et al. (2023). This dictionary includes 7 dimensions 'Budget', 'Cost', 'Investment', 'Operations', 'Performance', 'Risk', and 'Strategy' with 17 to 62 words/compound tokens each. In total there are 275 tokens. The dictionary contains the column 'Synonyms', i.e. all synonyms of certain seed words received via a Word-2-Vec model trained on Chinese annual reports. Its transformation is the column 'Synonyms_adjusted' which contains the tokens used for the analysis. The tansformations inlude: duplicates appearing in multiple dimensions are assigned to the dimension which fits the best to and too long tokens are split up, such that the longest token consists of a maximum of 3 words.

In [ ]:
#Import the dictionary with the columns 'Dimension', 'Synonyms', and 'Synonyms_adjusted'. 
map_dictionary = pd.read_csv('BoW/MAP_Dictionary_BoW_final.csv', sep=';')
map_dictionary['Dimension'] = map_dictionary['Dimension'].str.strip()

#Save the MAP tokens in a separat list-type object
MAP_tokens = list(map_dictionary['Synonyms_adjusted'])

#Have a look at the first entries of the dictionary
map_dictionary.head()

Second, we load the standard english web-trained small ('en_core_web_sm') spacy model/pipeline (disable 'tagger', 'parser', 'ner', 'textcat', and 'lemmatizer'). We use the model to create (compound) phrases/tokens in the 10-K filings. To do so we need to initialize a phrase-matcher and add the dictionary-based (compound) tokens to the matcher.

In [ ]:
# We just use a cpu model since we only need the tokenizer
spacy.require_cpu()

#Load the standard english web-trained small ('en_core_web_sm') spacy cpu model/pipeline (disable 'tagger', 'parser', 'ner', 'textcat', and 'lemmatizer')
nlp = spacy.load('en_core_web_sm', disable=['tagger', 'parser', 'ner', 'textcat', 'lemmatizer'])

#Since we just use the tokenizer we can increase the maximum length of a document withouth running into memory issues
nlp.max_length = 3000000

#Initialize PhraseMatcher and add the MAP phrase patterns / tokens.
matcher = spacy.matcher.PhraseMatcher(nlp.vocab)
patterns = [nlp.make_doc(token) for token in MAP_tokens]
matcher.add('PHRASES', patterns)

Third, we define a function that helps to count the occurence of (compound) tokens in a document based on a pre-defined dictionary

In [ ]:
def update_word_counts(doc, word_dict):
    matches = matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end].text
        if span in word_dict:
            word_dict[span] += 1

Fourth, we run a loop across all pre-processed filings and count the frequency of each MAP-related (compound) token in each filing. 

NOTE: This part takes some time (~3h on the authors local machine).

In [ ]:
#Create a new column in which the dictionary with the respective frequencies will be save in
corpus_df['MAP_token_count'] = None

#Run the loop to count the MAP-related (compound) tokens frequency
for doc in range(0,len(corpus_df['filing_text'])):

    if doc%50 == 0:
        print(f'{doc} documents have been processed')

    #Set counter to 0 for all tokens
    word_dict = {token: 0 for token in MAP_tokens}
    #Load the filing document into the spacy NLP pipeline
    doc_nlp = nlp(corpus_df.loc[doc,'filing_text'])
    #Update the word_dict with the word frequency of the considered document
    update_word_counts(doc_nlp, word_dict)
    #Save the dictionary with the word frequencies to the new column 'MAP_token_count' in corpus_df
    corpus_df.at[doc,'MAP_token_count'] = word_dict

#Delete the nlp object to free memory
del nlp, matcher, patterns, doc_nlp, word_dict

#Save the new dataframe with the count numbers stored in dataframe as a dictionary
corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_final.pkl')

<div class='alert-info'>
Step 2: Summarize the frequency of MAP-related (compound) tokens per dimension
</div>

In this step we will add up the (compound) token frequencies per dimension. To do so, we will loop through the dictionaries (including the (compound) tokens and their frequency) and then loop through the MAP dimensions to create a sum among the (compound) token frequencies included in the document dictionary that belong to the respective MAP dimension. We call these measures 'equally weighted' as each word is treated in the same way. We will adjust this approach at a later stage

In [ ]:
#First, we need to add new columns to our corpus_df in which we store the dimension frequency measures. 

corpus_df = corpus_df.reindex(columns=corpus_df.columns.tolist() + count_columns)

#Second, we loop through all filings and create the dimension measures
for doc in range(0,len(corpus_df['filing_text'])):

    if doc%50 == 0:
        print(f'{doc} documents have been processed')

    for dimension in map_dictionary['Dimension'].drop_duplicates().to_list():
        corpus_df.loc[doc,dimension + '_equally_weighted'] = sum(corpus_df.loc[doc,'MAP_token_count'].get(key, 0) for key in map_dictionary['Synonyms_adjusted'][map_dictionary['Dimension'] == dimension].to_list())

#Third, we save the new measures as integers
corpus_df[count_columns] = corpus_df[count_columns].astype(int)

#Fourth, save the dataframe with the raw MAP count variables

corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_final.pkl')

<div class='alert-info'>
Step 3: Create the final MAP measures used in the analysis
</div>

<div class='alert-info'>
Step 3.1: Equally-weighted MAP measures scaled by the total number of words
</div>

The first measure used in the analysis is the equally-weighted MAP measures scaled by the total number of words in the respective filing. We scale the frequencies by the total number of words to account for the length of the text (longer filings have a higher chance to include more MAP-related words). In addition, we will normalize the measure such that the final (equally-weighted) MAP score is between 0 and 1 (i.e., we substract the minimum and divide by the difference between max and min).

In total we creat 3 different kind of scores (repeat step 3.1 & 3.2 with the different settings):
1. MAP scores normalized across the whole sample including Financing/Investment firms (within_industry = False & without_finance = False)
2. MAP scores normalized across the whole sample excluding Financing/Investment firms (within_industry = False & without_finance = True)
3. MAP scores normalized within industry including Financing/Investment firms (within_industry = True & without_finance = False)

First, we load the cleaned corpus with the raw MAP count variables and specify whether we want to normalize the MAP measures within industry or across the whole sample. We also specify whether we want to exclude the Financing/Investment firms from the MAP measures creation (just necessary if within_industry=False).

In [ ]:
# specify whether to normalize MAP measures within industry (True) or across the whole sample (False)
within_industry = False  

# specify whether to exclude the Financing/Investment firms from the MAP measures creation (just necessary if within_industry=False)
without_finance = False

with open('BoW/Corpus_df_BoW_dimension_scores_final.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

# If within industry normalization is chosen, we delete all observations with empty industry information
if within_industry == True:
    corpus_df = corpus_df[corpus_df['NAICS Sector Name']!=''].copy().reset_index(drop=True)
    #drop everything after column 24
    corpus_df = corpus_df.iloc[:, :24]

# If without finance normalization is chosen, we delete all observations in the Finance and Insurance industry
if without_finance == True:
    corpus_df = corpus_df[corpus_df['NAICS Sector Name']!='Finance and Insurance'].copy().reset_index(drop=True)
    #drop everything after column 24
    corpus_df = corpus_df.iloc[:, :24]

First, we load the cleaned corpus with the raw MAP count variables and specify whether we want to normalize the MAP measures within industry or across the whole sample. We also specify whether we want to exclude the Financing/Investment firms from the MAP measures creation (just necessary if within_industry=False).

In [ ]:
count_columns = ['Strategy_equally_weighted', 'Budget_equally_weighted', 'Cost_equally_weighted', 
               'Operations_equally_weighted', 'Investment_equally_weighted', 'Performance_equally_weighted', 'Risk_equally_weighted']

# First, divide the number of map-related tokens by the total number of words per document 
for map_column in count_columns:
    corpus_df[map_column + '_scaled'] = corpus_df[map_column]/corpus_df['Word_count']

# Second, normalize the scaled MAP measures on a scale between 0 and 1 by substracting 
# the minimum and dividing it by the range between max and min within one dimension
# Note: As every dimension includes 0 as value substracting the minimum 
# does not make too much sense (Might want to exclude too short reports before)
map_scaled_columns = ['Strategy_equally_weighted_scaled', 'Budget_equally_weighted_scaled', 'Cost_equally_weighted_scaled', 
               'Operations_equally_weighted_scaled', 'Investment_equally_weighted_scaled', 'Performance_equally_weighted_scaled', 'Risk_equally_weighted_scaled']

# Third, we winsorize the weighted and scales MAP measures at the 1st and 99th percentile to avoid extreme outliers.
from scipy.stats.mstats import winsorize

# Apply Winsorization to selected columns
for col in map_scaled_columns:
    # Winsorize: limits=[lower limit %, upper limit %]
    corpus_df[col] = winsorize(corpus_df[col], limits=[0.01, 0.01])

# Fourth, loop through the columns and perform the normalization per column (alternative: per industry)
if within_industry == True:
    for industry in corpus_df['NAICS Sector Name'].unique():
        for map_column in map_scaled_columns:
            map_min = min(corpus_df[corpus_df['NAICS Sector Name']==industry][map_column])
            map_max = max(corpus_df[corpus_df['NAICS Sector Name']==industry][map_column])
            corpus_df.loc[corpus_df['NAICS Sector Name']==industry, map_column.split('_equally_weighted_scaled')[0] + '_equally_standardized'] = (corpus_df.loc[corpus_df['NAICS Sector Name']==industry, map_column] - map_min)/(map_max - map_min)
else: 
    for map_column in map_scaled_columns:
        map_min = min(corpus_df[map_column])
        map_max = max(corpus_df[map_column])
        corpus_df[map_column.split('_equally_weighted_scaled')[0] + '_equally_standardized'] = (corpus_df[map_column] - map_min)/(map_max - map_min)


# Fifth, save the dataframe with the new equally weighted and normalized MAP variables
if within_industry == True and without_finance == False:
    corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_within_industry_normalized.pkl')
elif within_industry == False and without_finance == True:
    corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_without_finance_normalized.pkl')
elif within_industry == False and without_finance == False:
    corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_final.pkl')
else:
    print('Please check your settings for within_industry and without_finance')

del map_scaled_columns, map_min, map_max

Last, we quickly check the result of the normalization and normalization of the MAP measures.

In [ ]:
#Have a look at the result
map_equally_standardized_columns = ['Strategy_equally_standardized', 'Budget_equally_standardized', 'Cost_equally_standardized', 
               'Operations_equally_standardized', 'Investment_equally_standardized', 'Performance_equally_standardized', 'Risk_equally_standardized']

display(corpus_df[map_equally_standardized_columns].head())

del map_equally_standardized_columns

<div class='alert-info'>
Step 3.2: TF-IDF-weighted MAP measures scaled by the total number of words
</div>

With our second measure we want to account for the frequency by which dimension-related words are occurring (some words are mentioned more often than others) and, as before, for the length of the document. To do so we calculate the Inverse Document Frequency (IDF) per token, the Term Frequency - Inverse Document Frequency (TF-IDF) measure per token, add up the TF-IDF measures per dimension, weight the TF-IDF measures per dimension with the length of filing, and finally normalize the TF-IDF score per dimension on a scale between 0 and 1.

In total we creat 3 different kind of scores:
1. MAP scores normalized across the whole sample including Financing/Investment firms (within_industry = False & without_finance = False)
2. MAP scores normalized across the whole sample excluding Financing/Investment firms (within_industry = False & without_finance = True)
3. MAP scores normalized within industry including Financing/Investment firms (within_industry = True & without_finance = False)

First, we will calculate the IDF per token. Doing so we use the following formula: $IDF(t) = log(\frac{N + 1}{1 + df(t)}) + 1$, where N is the total number of documents in the corpus, and df(t) is the raw count of documents in the corpus that include the token t.

In [ ]:
#Create the term matrix as a separate dataframe, i.e. a matrix that displays the frequency of the tokens from our dictionary 
# per document (columns are the tokens and rows the documents).
term_matrix = pd.DataFrame(dict(corpus_df['MAP_token_count'])).transpose()

#Initiaize an empty IDF dictionary
term_idf = {key: pd.NA for key in term_matrix.columns}

#Calculate the IDF per term/token
for term in term_matrix.columns:
    term_idf[term] = np.log((1+term_matrix[term].count())/(1+sum(term_matrix[term] > 0)))+1

#Have a look at the results 
print('The term matrix for a selection of words and the first 5 documents looks like this:')
      
display(term_matrix[['collaboration','supply chain','financing', 'internal risk control']].head())

print('The IDFs are:') 
print(f'''\'collaboration\' - {term_idf.get('collaboration', 'Key not found')}''')
print(f'''\'supply chain\' - {term_idf.get('supply chain', 'Key not found')}''')
print(f'''\'financing\' - {term_idf.get('financing', 'Key not found')}''')
print(f'''\'internal risk control\' - {term_idf.get('internal risk control', 'Key not found')}''')



Second, we multiply the term frequencies in each document by its respective IDF. Hence, we apply the following formula: $TF-IDF(t,d) = TF(t,d) * IDF(t)$ , where TF(t,d) is the term frequency of token t in document d, and IDF(t) the caclualted Inverse Document Frequency from the previous calcualtion.

In [ ]:
#Create an empty TF-IDF dataframe/matrix
tf_idf_matrix = pd.DataFrame(columns=term_matrix.columns)

#Loop through the columns (tokens) of term matrix and multiply each column (token) with its IDF
for term in term_matrix.columns:
    tf_idf_matrix[term] = term_matrix[term].copy() * term_idf[term]

#Create a new column 'MAP_token_tf_idf' in the corpus dataframe and save the TF-IDF values 
# of the tokens per document as a dictionary into it
corpus_df['MAP_token_tf_idf'] = pd.NA

corpus_df['MAP_token_tf_idf'] = tf_idf_matrix.apply(lambda row: row.to_dict(), axis=1)

#Have a look at the results
print('The TF-IDF matrix for a selection of words and the first 5 documents looks like this:')
      
display(tf_idf_matrix[['collaboration','supply chain','financing', 'internal risk control']].head())

del tf_idf_matrix

Third, we calculate the TF-IDF measures per MAP dimension by adding up the TF-IDF measure of tokens beloging to the respective MAP dimension at the filing level.  

In [ ]:
#Loop through each document and within each document loop through the MAP dimensions and add up the respective TF-IDF weighted tokens
for doc in range(0,len(corpus_df['filing_text'])):

    if doc%50 == 0:
        print(f'{doc} documents have been processed')

    for dimension in map_dictionary['Dimension'].drop_duplicates().to_list():
        corpus_df.loc[doc,dimension + '_tf_idf_weighted'] = sum(corpus_df.loc[doc,'MAP_token_tf_idf'].get(key, 0) for key in map_dictionary['Synonyms_adjusted'][map_dictionary['Dimension'] == dimension].to_list())

Fourth, we divide the TF-IDF weighted token/word frequencies by the total number of words in the respective document.

In [ ]:
#Define the columns of the TF-IDF weighted dimensions we calcualted in the previous step
tf_idf_columns = ['Strategy_tf_idf_weighted', 'Budget_tf_idf_weighted',
       'Cost_tf_idf_weighted', 'Operations_tf_idf_weighted',
       'Investment_tf_idf_weighted', 'Performance_tf_idf_weighted',
       'Risk_tf_idf_weighted']

#Loop through the columns and divide the TF-IDF weighted dimensions by the total number of words per document 
for map_column in tf_idf_columns:
    corpus_df[map_column + '_scaled'] = corpus_df[map_column]/corpus_df['Word_count']

del tf_idf_columns

Fifth, we normalize the TF-IDF scores such that the final (TF-IDF weighted) MAP score is between 0 and 1 (i.e., we substract the minimum and divide by the difference between max and min).

In [ ]:
map_tf_idf_scaled_columns = ['Strategy_tf_idf_weighted_scaled', 'Budget_tf_idf_weighted_scaled',
       'Cost_tf_idf_weighted_scaled', 'Operations_tf_idf_weighted_scaled',
       'Investment_tf_idf_weighted_scaled', 'Performance_tf_idf_weighted_scaled',
       'Risk_tf_idf_weighted_scaled']

#First, we winsorize the weighted and scales MAP measures at the 1st and 99th percentile to avoid extreme outliers.

# Apply Winsorization to selected columns
for col in map_tf_idf_scaled_columns:
    # Winsorize: limits=[lower limit %, upper limit %]
    corpus_df[col] = winsorize(corpus_df[col], limits=[0.01, 0.01])

#Second, loop through the columns and perform the normalization per column (alternative: per industry)
if within_industry == True:
    for industry in corpus_df['NAICS Sector Name'].unique():
        for map_column in map_tf_idf_scaled_columns:
            map_min = min(corpus_df[corpus_df['NAICS Sector Name']==industry][map_column])
            map_max = max(corpus_df[corpus_df['NAICS Sector Name']==industry][map_column])
            corpus_df.loc[corpus_df['NAICS Sector Name']==industry, map_column.split('_tf_idf_weighted_scaled')[0] + '_tf_idf_standardized'] = (corpus_df.loc[corpus_df['NAICS Sector Name']==industry, map_column] - map_min)/(map_max - map_min)
else:
    for map_column in map_tf_idf_scaled_columns:
        map_min = min(corpus_df[map_column])
        map_max = max(corpus_df[map_column])
        corpus_df[map_column.split('_tf_idf_weighted_scaled')[0] + '_tf_idf_standardized'] = (corpus_df[map_column] - map_min)/(map_max - map_min)

#Third, save the final data as pickle format
if within_industry == True and without_finance == False:
    corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_within_industry_normalized.pkl')
elif within_industry == False and without_finance == True:
    corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_without_finance_normalized.pkl')
elif within_industry == False and without_finance == False:
    corpus_df.to_pickle('BoW/Corpus_df_BoW_dimension_scores_final.pkl')
else:
    print('Please check your settings for within_industry and without_finance')

del map_tf_idf_scaled_columns, map_min, map_max

In [ ]:
#Have a look at the result
map_tf_idf_standardized_columns = ['Strategy_tf_idf_standardized', 'Budget_tf_idf_standardized', 'Cost_tf_idf_standardized', 
               'Operations_tf_idf_standardized', 'Investment_tf_idf_standardized', 'Performance_tf_idf_standardized', 'Risk_tf_idf_standardized']

display(corpus_df[map_tf_idf_standardized_columns].head())

del map_tf_idf_standardized_columns

## Descriptive Analyses of MAP Dimension Scores

In this section, descriptive analyses are performed. First, we will have a look at the raw MAP token-count variables per MAP dimension, i.e. evaluate the occurence of the MAP-related words in the corpus, including their development of mean over time, and the difference in means across industries. Second, the equally weighted and scaled MAP measures are investigated. Last, we perfom the same analyses for the TF-IDF weigthed and scaled MAP measures. 

<div class='alert-warning'>
Load the corpus dataframe (in case you have not done it yet)
</div>

In [ ]:
# specify whether to normalize MAP measures within industry (True) or across the whole sample (False)
within_industry = False
# specify whether to exclude the Financing/Investment firms from the MAP measures creation (just necessary if within_industry=False)
without_finance = True

#Load the final BoW corpus
if within_industry == True and without_finance == False:
    with open('BoW/Corpus_df_BoW_dimension_scores_within_industry_normalized.pkl','rb') as path_name:
        corpus_df = pickle.load(path_name)
elif within_industry == False and without_finance == True:
    with open('BoW/Corpus_df_BoW_dimension_scores_without_finance_normalized.pkl','rb') as path_name:
        corpus_df = pickle.load(path_name)
elif within_industry == False and without_finance == False:
    with open('BoW/Corpus_df_BoW_dimension_scores_final.pkl','rb') as path_name:
        corpus_df = pickle.load(path_name)
else:
    print('Please check your settings for within_industry and without_finance')

#Define the columns in the dataframe that include the information on the raw MAP measures
count_columns = ['Strategy_equally_weighted', 'Budget_equally_weighted', 'Cost_equally_weighted', 
               'Operations_equally_weighted', 'Investment_equally_weighted', 'Performance_equally_weighted', 'Risk_equally_weighted']

#Define the columns in the dataframe that include the information on the equally weighted & normalized MAP measures
equally_columns =['Strategy_equally_standardized', 'Budget_equally_standardized', 
                      'Cost_equally_standardized', 'Operations_equally_standardized', 
                      'Investment_equally_standardized', 'Performance_equally_standardized', 
                      'Risk_equally_standardized']

#Define the columns in the dataframe that include the information on the TF-IDF weighted & normalized MAP measures
tf_idf_columns = ['Strategy_tf_idf_standardized', 'Budget_tf_idf_standardized',
       'Cost_tf_idf_standardized', 'Operations_tf_idf_standardized',
       'Investment_tf_idf_standardized', 'Performance_tf_idf_standardized',
       'Risk_tf_idf_standardized']

analysis_df = corpus_df[['filing_key', 'filing_year', 'NAICS Sector Name', 'NAICS Subsector Name'] + count_columns + equally_columns + tf_idf_columns].copy()

del corpus_df

<div class='alert-warning'>
Set output path and size for figures/plots
</div>

In [ ]:
#Directory to save plots
output_dir_plots_raw = 'Analyses_outputs/Plots/MAP_Dimension_Scores/BoW/Raw count'
if not os.path.exists(output_dir_plots_raw):
    os.makedirs(output_dir_plots_raw)

output_dir_plots_equally = 'Analyses_outputs/Plots/MAP_Dimension_Scores/BoW/Equally_weighted_normalized'
if not os.path.exists(output_dir_plots_equally):
    os.makedirs(output_dir_plots_equally)

output_dir_plots_tf_idf =  'Analyses_outputs/Plots/MAP_Dimension_Scores/BoW/TF_IDF_weighted_normalized'
if not os.path.exists(output_dir_plots_tf_idf):
    os.makedirs(output_dir_plots_tf_idf)

#Define the figure size for the output
plotnine.options.figure_size = (12,7)

<div class='alert-info'>
Step 1: Raw Term Count Analyses
</div>

In the first analysis step, we have a look at the raw MAP dimension measures, i.e. how often the correspoding MAP-related tokens appear in the corpus. We use the previously created variables '..._equally_weigthed' to perform the analyses. As mentioned before, we look at a summary statistic, the total frequency of words/tokens per dimension, the development of the mean over time, and the means across industries.

First, we have a look at the summary statistics of the raw word count per dimension.

In [ ]:
#1. Print the summary statistic
display(analysis_df[count_columns].describe())

Second, we have a look at the total frequency of MAP-related words per dimension and save it as frequency plot.


In [ ]:
#2. We create a figure that displays the total sum for each raw MAP measure

#Frist, we sum up the total frequency of tokens per dimension
total_frequencies = analysis_df[count_columns].sum().reset_index()

#Second, we rename the columns and the dimensions
total_frequencies.columns = ['Dimension','Frequency']
total_frequencies['Dimension'] = total_frequencies['Dimension'].apply(lambda x: x.split('_')[0])

#Third, we save the total frequency (sum per dimension) as integer value 
total_frequencies['Frequency'] = total_frequencies['Frequency'].astype(int)

#Fourth, create a new column in the DataFrame with comma formatting
total_frequencies['Formatted_Frequency'] = total_frequencies['Frequency'].apply(lambda x: f'{x:,}')

#Fourh create the plot
plot = (plotnine.ggplot(total_frequencies, plotnine.aes(
    x='reorder(Dimension, Frequency)', 
    y='Frequency',
    label='Formatted_Frequency',  # Apply comma formatting 
    fill='Dimension'))
    # We define that the length of the bar equals the total frequency of the respective dimension
    + plotnine.geom_bar(stat='identity') 
    # We swap the x and y-axis 
    + plotnine.coord_flip()
    # We change the x-axis label
    + plotnine.xlab('MAP Dimension')
    # ... and the y-axis label
    + plotnine.ylab('Total Token Frequency')
    # We add the number of the respective frequency at the right end of the bar 
    + plotnine.geom_text(ha='left', va='center', color='black', size=18, fontweight='bold'  )
    # Format y-axis with commas as thousand separators and adjust the limits such that the labels are displayed properly
    + plotnine.scale_y_continuous(labels=comma_format(), limits=(0, 600000))
    # Change the appearance (white background, bold title, etc..)
    + plotnine.theme(
        legend_position='none',
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold'),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(size=12, weight='bold'),  
        axis_text_y=plotnine.element_text(size=12, weight='bold')
    )
    
)

#Fifth, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_raw, 'Word_Frequency_Plot_Raw_MAP_Dimensions_within_industry_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_raw, 'Word_Frequency_Plot_Raw_MAP_Dimensions_without_finance_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_raw, 'Word_Frequency_Plot_Raw_MAP_Dimensions.png'), width=19.2, height=9.67, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')

del total_frequencies
#Last, display the plot
plot.draw()

Third, we investigate the development of the means over time.

In [ ]:
#3. Create a figure that display the development of the token frequency for each dimension (here: one with all dimensions in one figure)

#First, we group the MAP dimensions by filing year and calculate the average per dimension and filing year
MAP_dimensions_year_avg_df = analysis_df.groupby('filing_year')[count_columns].mean().reset_index()

#Second, we rename the variables for which the plot should be created
MAP_dimensions_year_avg_df.columns = ['filing_year', 'Strategy', 'Budget','Cost', 'Operations', 'Investment', 'Performance', 'Risk']

#Third, we transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
MAP_dimensions_year_avg_df_long = MAP_dimensions_year_avg_df.melt(id_vars=['filing_year'], var_name='Dimension', value_name='Average')

#Fourth, we define the range of years in the corpus (needed to set axis limits in the plot)
years = MAP_dimensions_year_avg_df_long['filing_year'].unique()
year_min = years.min()
year_max = years.max()

#Fifth, create one plot with all variables
plot = (plotnine.ggplot(MAP_dimensions_year_avg_df_long, plotnine.aes(x='filing_year', y='Average', color='Dimension', group='Dimension')) 
    + plotnine.geom_line(size=1.5) 
    + plotnine.theme_minimal() 
    + plotnine.labs(
         x='Filing Year',
         y='Average Token Frequency')
    + plotnine.scale_x_continuous(breaks=range(year_min, year_max + 1))
    + plotnine.scale_y_continuous(breaks=range(0, 121, 30), limits=(0, 120))
    + plotnine.theme(
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold'),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(size=14, weight='bold'),  
        axis_text_y=plotnine.element_text(size=14, weight='bold')
    )
)

#Seventh, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_raw, 'Development_Frequency_Raw_MAP_Dimensions_within_industry_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_raw, 'Development_Frequency_Raw_MAP_Dimensions_without_finance_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_raw, 'Development_Frequency_Raw_MAP_Dimensions.png'), width=19.2, height=9.67, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')


del MAP_dimensions_year_avg_df_long, years

#Last, display the plot
plot.draw()

Fourth, we analyse the means across NAICS industries.

In [ ]:
#4. Create a plot that shows the average token frequncy per industry and dimension (here: one with all dimensions in one figure)

#First, define the columns we need for the analysis
columns = ['filing_year', 'NAICS Sector Name'] + count_columns

#Second, transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
MAP_dimension_industry_long_df = analysis_df[columns].melt(id_vars=['filing_year','NAICS Sector Name'],var_name='Dimension', value_name='Frequency')

#Third, calculate the token frequency mean for each group (Industry,Dimension)
MAP_dimension_industry_long_avg_df = MAP_dimension_industry_long_df.groupby(['NAICS Sector Name', 'Dimension']).agg({'Frequency': 'mean'}).reset_index()

#Fourth, rename the variable values (cutoff: '_frequency')
MAP_dimension_industry_long_avg_df['Dimension'] = MAP_dimension_industry_long_avg_df['Dimension'].apply(lambda x: x.split('_')[0])

#Fifth, create one plot with all variables
plot = (
    plotnine.ggplot(MAP_dimension_industry_long_avg_df, plotnine.aes(x='NAICS Sector Name', y='Frequency', color='Dimension', group='Dimension'))
    + plotnine.geom_point(size=4)
    + plotnine.geom_line(size=1)
    + plotnine.xlab('Industry')
    + plotnine.ylab('Average Token Frequency')
    + plotnine.scale_y_continuous(limits=(0, MAP_dimension_industry_long_avg_df['Frequency'].max() + 30))
    + plotnine.theme(
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold'),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
        axis_text_y=plotnine.element_text(size=14, weight='bold')
    )
)

#Sixth, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_raw, 'Raw_MAP_Dimensions_Frequency_per_Industry_within_industry_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_raw, 'Raw_MAP_Dimensions_Frequency_per_Industry_without_finance_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_raw, 'Raw_MAP_Dimensions_Frequency_per_Industry.png'), width=19.2, height=9.67, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')

del MAP_dimension_industry_long_df

#Last, display the plot
plot.draw()

<div class='alert-info'>
Step 2: Equally Weighted and Normalized MAP Measures Analyses 
</div>

Now, we have a look at the equally weighted and normalized MAP measures, i.e. each measures is a equally weighted sum of the respective tokens, divided by the total number of words in the filing, and normalized (such that they are between 0 and 1). We use the previously created variables '..._equally_standardized' to perform the analyses (they constitut a kind of 'weighted term matrix' at the MAP dimension level). As mentioned before we look at a summary statistic, a distribution plot (BoxPlot), the development of the mean over time, and the means across industries.

First, we have a look at the summary statistics of the equally weighted and normalized MAP measure per dimension.

In [ ]:
#Print the summary statistic
display(analysis_df[equally_columns].describe())

Second, we investigate the development of the means over time.

In [ ]:
#2. Create a figure that display the development of the equally weigthed and normalized MAP measures (here: one with all dimensions in one figure)

#First, we group the MAP dimensions by filing year and calculate the average per dimension and filing year
MAP_dimensions_equally_year_avg_df = analysis_df.groupby('filing_year')[equally_columns].mean().reset_index()

#Second, we rename the variables for which the plot should be created
MAP_dimensions_equally_year_avg_df.columns = ['filing_year', 'Strategy', 'Budget','Cost', 'Operations', 'Investment', 'Performance', 'Risk']

#Third, we transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
MAP_dimensions_equally_year_avg_df_long = MAP_dimensions_equally_year_avg_df.melt(id_vars=['filing_year'], var_name='Dimension', value_name='Average')

#Fourth, we define the range of years/MAP averages in the corpus (needed to set axis limits in the plot)
years = MAP_dimensions_equally_year_avg_df_long['filing_year'].unique()
year_min = years.min()
year_max = years.max()
map_max = np.round(MAP_dimensions_equally_year_avg_df_long['Average'].max(), 1)

#Fifth, create one plot with all variables
plot = (plotnine.ggplot(MAP_dimensions_equally_year_avg_df_long, plotnine.aes(x='filing_year', y='Average', color='Dimension', group='Dimension')) 
    + plotnine.geom_line(size=1.5) 
    + plotnine.theme_minimal() 
    + plotnine.labs(
         x='Filing Year',
         y='Average MAP Measure')
    + plotnine.scale_x_continuous(breaks=range(year_min, year_max + 1))
    + plotnine.scale_y_continuous(breaks=np.arange(0, map_max+0.05, 0.1), limits=(0, map_max+0.05))
    + plotnine.theme(
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold'),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(rotation=90, hjust=1, size=14, weight='bold'),  
        axis_text_y=plotnine.element_text(size=14, weight='bold')
    )
)

#Sixth, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_equally, 'Development_Equally_Standardized_MAP_Dimensions_within_industry_normalized.png'), width=15.1, height=7.96, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_equally, 'Development_Equally_Standardized_MAP_Dimensions_without_finance_normalized.png'), width=15.1, height=7.96, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_equally, 'Development_Equally_Standardized_MAP_Dimensions.png'), width=15.1, height=7.96, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')

del MAP_dimensions_equally_year_avg_df_long, years

#Last, display the plot
plot.draw()

Third, we analyse the means across industries.

In [ ]:
#3. Create a plot that shows the equally weigthed and normalized MAP dimensions per industry (here: one with all dimensions in one figure)

#First, define the columns we need for the analysis and create a subset of the analysis dataframe with firms that have a valid industry classification
columns = ['filing_year', 'NAICS Sector Name'] + equally_columns

#Second, transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
MAP_dimension_equally_industry_long_df = analysis_df[columns].melt(id_vars=['filing_year','NAICS Sector Name'],var_name='Dimension', value_name='Average')

#Third, calculate the equally weigthed and normalized MAP dimension mean for each group (Industry,Dimension)
MAP_dimension_equally_industry_long_avg_df = MAP_dimension_equally_industry_long_df.groupby(['NAICS Sector Name', 'Dimension']).agg({'Average': 'mean'}).reset_index()

#Fourth, rename the variable values (cutoff: '_frequency')
MAP_dimension_equally_industry_long_avg_df['Dimension'] = MAP_dimension_equally_industry_long_avg_df['Dimension'].apply(lambda x: x.split('_')[0])

#Fifth, create one plot with all variables
plot = (
    plotnine.ggplot(MAP_dimension_equally_industry_long_avg_df, plotnine.aes(x='NAICS Sector Name', y='Average', color='Dimension', group='Dimension'))
    + plotnine.geom_point(size=4)
    + plotnine.geom_line(size=1)
    + plotnine.xlab('Industry')
    + plotnine.ylab('Average MAP Measure')
    + plotnine.scale_y_continuous(breaks=np.arange(0, MAP_dimension_equally_industry_long_avg_df['Average'].max(), 0.1), limits=(0, MAP_dimension_equally_industry_long_avg_df['Average'].max()))
    + plotnine.theme(
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
        axis_text_y=plotnine.element_text(size=14, weight='bold')
    )
)

#Sixth, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_equally, 'MAP_Dimensions_Equally_Standardized_per_Industry_within_industry_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_equally, 'MAP_Dimensions_Equally_Standardized_per_Industry_without_finance_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_equally, 'MAP_Dimensions_Equally_Standardized_per_Industry.png'), width=19.2, height=9.67, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')

del MAP_dimension_equally_industry_long_df

#Last, display the plot
plot.draw()

<div class='alert-info'>
Step 3: TF-IDF Weighted and Normalized MAP Measures Analyses 
</div>

Now, we have a look at the TF-IDF weighted and normalized MAP measures, i.e. each measures is a TF-IDF weighted sum of the respective tokens, divided by the total number of words in the filing, and normalized (such that they are between 0 and 1). We use the previously created variables '..._tf_idf_standardized' to perform the analyses (they constitut a kind of 'weighted term matrix' at the MAP dimension level). As mentioned before we look at a summary statistic, a distribution plot (BoxPlot), the development of the mean over time, and the means across industries.

First, we have a look at the summary statistics of the CS weighted and normalized MAP measure per dimension.

In [ ]:
#1. We create a summary statistic for the TF-IDF weigthed and normalized MAP measures
display(analysis_df[tf_idf_columns].describe())

Second, we investigate the development of the means over time.

In [ ]:
#2. Create a figure that display the development of the TF-IDF weigthed and normalized MAP measures (here: one with all dimensions in one figure)

#First, we group the MAP dimensions by filing year and calculate the average per dimension and filing year
MAP_dimensions_tf_idf_year_avg_df = analysis_df.groupby('filing_year')[tf_idf_columns].mean().reset_index()

#Second, we rename the variables for which the plot should be created
MAP_dimensions_tf_idf_year_avg_df.columns = ['filing_year', 'Strategy', 'Budget','Cost', 'Operations', 'Investment', 'Performance', 'Risk']

#Third, we transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
MAP_dimensions_tf_idf_year_avg_df_long = MAP_dimensions_tf_idf_year_avg_df.melt(id_vars=['filing_year'], var_name='Dimension', value_name='Average')

#Fourth, we define the range of years/MAP averages in the corpus (needed to set axis limits in the plot)
years = MAP_dimensions_tf_idf_year_avg_df_long['filing_year'].unique()
year_min = years.min()
year_max = years.max()
map_max = np.round(MAP_dimensions_tf_idf_year_avg_df_long['Average'].max(), 1)

#Fifth, create one plot with all variables
plot = (plotnine.ggplot(MAP_dimensions_tf_idf_year_avg_df_long, plotnine.aes(x='filing_year', y='Average', color='Dimension', group='Dimension')) 
    + plotnine.geom_line(size=1.5) 
    + plotnine.theme_minimal() 
    + plotnine.labs(
         x='Filing Year',
         y='Average MAP Measure')
    + plotnine.scale_x_continuous(breaks=range(year_min, year_max + 1))
    + plotnine.scale_y_continuous(breaks=np.arange(0, map_max+0.05, 0.1), limits=(0, map_max+0.05))
    + plotnine.theme(
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold'),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(size=12, weight='bold'),  
        axis_text_y=plotnine.element_text(size=12, weight='bold')
    )
)

#Sixth, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_tf_idf, 'Development_TF_IDF_Standardized_MAP_Dimensions_within_industry_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_tf_idf, 'Development_TF_IDF_Standardized_MAP_Dimensions_without_finance_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_tf_idf, 'Development_TF_IDF_Standardized_MAP_Dimensions.png'), width=19.2, height=9.67, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')

del MAP_dimensions_tf_idf_year_avg_df_long, years

#Last, display the plot
plot.draw()

Third, we analyse the means across industries.

In [ ]:
#3. Create a plot that shows the TF-IDF weigthed and normalized MAP dimensions per industry (here: one with all dimensions in one figure)

#First, define the columns we need for the analysis and create a subset of the analysis dataframe with firms that have a valid industry classification
columns = ['filing_year', 'NAICS Sector Name'] + tf_idf_columns

#Second, transform the dataframe to a long format (1 filing has 7 rows - one for each dimension)
MAP_dimension_tf_idf_industry_long_df = analysis_df[columns].melt(id_vars=['filing_year','NAICS Sector Name'],var_name='Dimension', value_name='Average')

#Third, calculate the TF-IDF weigthed and normalized MAP dimension mean for each group (Industry,Dimension)
MAP_dimension_tf_idf_industry_long_avg_df = MAP_dimension_tf_idf_industry_long_df.groupby(['NAICS Sector Name', 'Dimension']).agg({'Average': 'mean'}).reset_index()

#Fourth, rename the variable values (cutoff: '_frequency')
MAP_dimension_tf_idf_industry_long_avg_df['Dimension'] = MAP_dimension_tf_idf_industry_long_avg_df['Dimension'].apply(lambda x: x.split('_')[0])

#Fifth, create one plot with all variables
plot = (
    plotnine.ggplot(MAP_dimension_tf_idf_industry_long_avg_df, plotnine.aes(x='NAICS Sector Name', y='Average', color='Dimension', group='Dimension'))
    + plotnine.geom_point(size=4)
    + plotnine.geom_line(size=1)
    + plotnine.xlab('Industry')
    + plotnine.ylab('Average MAP Measure')
    + plotnine.scale_y_continuous(breaks=np.arange(0, MAP_dimension_tf_idf_industry_long_avg_df['Average'].max() + 0.1, 0.1), limits=(0, MAP_dimension_tf_idf_industry_long_avg_df['Average'].max()+0.1))
    + plotnine.theme(
        panel_background= plotnine.element_rect(fill='white'),
        plot_background= plotnine.element_rect(fill='white'), 
        plot_title=plotnine.element_text(size=22, weight='bold'),  
        axis_title_x=plotnine.element_text(size=18, weight='bold', vjust=-5),  
        axis_title_y=plotnine.element_text(size=18, weight='bold'),  
        legend_title=plotnine.element_text(size=18),
        legend_text= plotnine.element_text(size=16),
        axis_text_x=plotnine.element_text(rotation=45, hjust=1, size=14, weight='bold'),  
        axis_text_y=plotnine.element_text(size=14, weight='bold')
    )
)

#Sixth, save the plot to the ouput directory 
if within_industry == True and without_finance == False:
    plot.save(os.path.join(output_dir_plots_tf_idf, 'MAP_Dimensions_TF_IDF_Standardized_per_Industry_within_industry_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == True:
    plot.save(os.path.join(output_dir_plots_tf_idf, 'MAP_Dimensions_TF_IDF_Standardized_per_Industry_without_finance_normalized.png'), width=19.2, height=9.67, dpi=300)
elif within_industry == False and without_finance == False:
    plot.save(os.path.join(output_dir_plots_tf_idf, 'MAP_Dimensions_TF_IDF_Standardized_per_Industry.png'), width=19.2, height=9.67, dpi=300)
else:
    print('Please check your settings for within_industry and without_finance')

del MAP_dimension_tf_idf_industry_long_df

#Last, display the plot
plot.draw()

## Assessing Classification Performance 


In this part, we will use the evaluation dataset created for the GLLM approach (assessing performance of GLLMs and training) and have a look how the dictionary method would perform on the same dataset. Note that we will just use the validation dataset and not the part we used for fine-tuning.

First, import the validation dataset and the MAP dictionary.

In [ ]:
# Import the validation dataset (validation_data_MAP_sentences.xlsx)
validation_df = pd.read_excel('GLLM/Fine_tuning_data/validation_data_MAP_sentences.xlsx')

#Import the dictionary with the columns 'Dimension', 'Synonyms', and 'Synonyms_adjusted'. 
map_dictionary = pd.read_csv('BoW/MAP_Dictionary_BoW_final.csv', sep=';')
map_dictionary['Dimension'] = map_dictionary['Dimension'].str.strip()

#Save the MAP tokens in a separat list-type object
MAP_tokens = list(map_dictionary['Synonyms_adjusted'])

# replace the 'dimension' names with the corresponding names in the validation dataset
map_dictionary['Dimension'] = map_dictionary['Dimension'].replace({
    'Strategy': 'Strategy',
    'Budget': 'Budgeting / Planning',
    'Cost': 'Cost',
    'Operations': 'Operations',
    'Investment': 'Financing / Investment',
    'Performance': 'Performance / Internal Reporting',
    'Risk': 'Risk / Internal Control'
})

Second, we load the standard english web-trained large ('en_core_web_lg') spacy model/pipeline (disable 'tagger', 'parser', 'ner', 'textcat', and 'lemmatizer'). We use the model to create count the MAP(compound) phrases/tokens in the sentences. To do so we need to initialize a phrase-matcher and add the dictionary-based (compound) tokens to the matcher.

In [ ]:
# We just use a cpu model since we only need the tokenizer
spacy.require_cpu()

#Load the standard english web-trained small ('en_core_web_sm') spacy cpu model/pipeline (disable 'tagger', 'parser', 'ner', 'textcat', and 'lemmatizer')
nlp = spacy.load('en_core_web_sm', disable=['tagger', 'parser', 'ner', 'textcat', 'lemmatizer'])

#Since we just use the tokenizer we can increase the maximum length of a document withouth running into memory issues
nlp.max_length = 3000000

#Initialize PhraseMatcher and add the MAP phrase patterns / tokens.
matcher = spacy.matcher.PhraseMatcher(nlp.vocab)
patterns = [nlp.make_doc(token) for token in MAP_tokens]
matcher.add('PHRASES', patterns)

Third, we define a function that helps to count the occurence of MAP (compound) tokens in a sentence based on a pre-defined dictionary

In [ ]:
def update_word_counts(sentence, word_dict):
    matches = matcher(sentence)
    for match_id, start, end in matches:
        span = sentence[start:end].text
        if span in word_dict:
            word_dict[span] += 1

Fourth, we run a loop across all sentences and count the frequency of each MAP-related (compound) token in each sentence. 

In [ ]:
#Create a new column in which the dictionary with the respective frequencies will be save in
validation_df['MAP_token_count'] = None

#Run the loop to count the MAP-related (compound) tokens frequency
for sentence in range(0,len(validation_df['Sentence'])):

    #Set counter to 0 for all tokens
    word_dict = {token: 0 for token in MAP_tokens}
    #Load the filing document into the spacy NLP pipeline
    doc_nlp = nlp(validation_df.loc[sentence,'Sentence'].lower())
    #Update the word_dict with the word frequency of the considered document
    update_word_counts(doc_nlp, word_dict)
    #Save the dictionary with the word frequencies to the new column 'MAP_token_count' in validation_df
    validation_df.at[sentence,'MAP_token_count'] = word_dict

#Delete the nlp object to free memory
del nlp

#Save the new dataframe with the count numbers
validation_df.to_excel(f'BoW/Intermediate_datasets/validation_set_with_MAP_token_counts_BoW.xlsx', index=False)

Fifth, now we can create the column 'BoW_Implicit_MAP_referral' and set this to 'Yes' if at least on word of the MAP dictionary occurs in the sentence. In addition, we create a column 'BoW_Dimension'. For each row, we store a list of MAP dimensions that have at least one word that occur in the sentence. 

In [ ]:
# Create new column 'BoW_Implicit_MAP_referral' in validation_df to store the BoW implicit MAP referral and 'BoW_Dimension' to store the BoW MAP dimension
validation_df['BoW_MAP_referral'] = 'No'
validation_df['BoW_Dimension'] = None

# Loop through each row in the validation_df to determine if there is an implicit MAP referral and the corresponding dimensions
for index, row in validation_df.iterrows():
    word_count_dict = row['MAP_token_count']
    total_map_tokens = sum(word_count_dict.values())
    if total_map_tokens > 0:
        validation_df.at[index, 'BoW_MAP_referral'] = 'Yes'
        dimensions_found = set()
        for token, count in word_count_dict.items():
            if count > 0:
                dimension = map_dictionary[map_dictionary['Synonyms_adjusted'] == token]['Dimension'].values[0]
                dimensions_found.add(dimension)
        validation_df.at[index, 'BoW_Dimension'] = ', '.join(dimensions_found)

#Save the updated dataframe
validation_df.to_excel(f'BoW/Intermediate_datasets/validation_set_with_MAP_token_counts_BoW.xlsx', index=False)

Sixth, we define a fuction to evaluate the performance of the BoW approach and run the evaluation.

In [ ]:
def evaluate_Bow(dataset, truth_exp_col='Explicit_MAP_referral', pred_exp_col='BoW_MAP_referral',
                 
                          truth_imp_col='Implicit_MAP_referral', pred_imp_col='BoW_MAP_referral',
                          truth_dimension_col='MAP_dimension_1', pred_dimension_col='BoW_Dimension'):
    
    #Evaluate Explicit MAP Referral
    print('=== Explicit MAP Referral Evaluation ===')
    if not dataset.empty:
        exp_accuracy = accuracy_score(dataset[truth_exp_col], dataset[pred_exp_col])
        exp_f1_yes = f1_score(dataset[truth_exp_col], dataset[pred_exp_col], pos_label='Yes', zero_division=0)
        exp_precision_yes = precision_score(dataset[truth_exp_col], dataset[pred_exp_col], pos_label='Yes', zero_division=0)
        exp_recall_yes = recall_score(dataset[truth_exp_col], dataset[pred_exp_col], pos_label='Yes', zero_division=0)
        exp_f1_no = f1_score(dataset[truth_exp_col], dataset[pred_exp_col], pos_label='No', zero_division=0)
        exp_precision_no = precision_score(dataset[truth_exp_col], dataset[pred_exp_col], pos_label='No', zero_division=0)
        exp_recall_no = recall_score(dataset[truth_exp_col], dataset[pred_exp_col], pos_label='No', zero_division=0)
        print('\nClassification Report (Explicit):')
        print(classification_report(dataset[truth_exp_col], dataset[pred_exp_col], labels=['Yes', 'No'], zero_division=0, digits=3))
    else:
        print('No valid rows for Explicit MAP evaluation.')
    
    #Evaluate Implicit MAP Referral
    print('\n=== Implicit MAP Referral Evaluation ===')
    if not dataset.empty:
        imp_accuracy = accuracy_score(dataset[truth_imp_col], dataset[pred_imp_col])
        imp_precision_yes = precision_score(dataset[truth_imp_col], dataset[pred_imp_col], pos_label='Yes', zero_division=0)
        imp_recall_yes = recall_score(dataset[truth_imp_col], dataset[pred_imp_col], pos_label='Yes', zero_division=0)
        imp_f1_yes = f1_score(dataset[truth_imp_col], dataset[pred_imp_col], pos_label='Yes', zero_division=0)
        imp_precision_no = precision_score(dataset[truth_imp_col], dataset[pred_imp_col], pos_label='No', zero_division=0)
        imp_recall_no = recall_score(dataset[truth_imp_col], dataset[pred_imp_col], pos_label='No', zero_division=0)
        imp_f1_no = f1_score(dataset[truth_imp_col], dataset[pred_imp_col], pos_label='No', zero_division=0)
        print('\nClassification Report (Implicit):')
        print(classification_report(dataset[truth_imp_col], dataset[pred_imp_col], labels=['Yes', 'No'], zero_division=0, digits=3))
    else:
        print('No valid rows for Implicit MAP evaluation.')

    #Evaluate MAP Dimension
    print('\n=== MAP Dimension Evaluation ===')
    filtered_dimension = dataset[
        (dataset['BoW_MAP_referral'] == 'No') &
        (~dataset['BoW_Dimension'].isna())
    ]
    dim_percentage = len(filtered_dimension)/len(dataset)*100
    print(f"Number of rows where LLM says 'No' to both Explicit and Implicit MAP referral but MAP Dimension is not None: {dim_percentage:.0f}%")

    # Create a list of all dimension columns
    dimension_cols = [col for col in dataset.columns if col.startswith('MAP_dimension')]
    
    # Check if the micro / macro F1 score for the dimension column 
    label_space = ['Budgeting / Planning', 'Cost', 'Financing / Investment', 'Operations', 'Performance / Internal Reporting', 'Risk / Internal Control', 'Strategy', 'Pricing & Revenue Management']
    
    def encode_labels(text, label_space):
        labels = [l.strip() for l in text.split(',')]
        return [1 if label in labels else 0 for label in label_space]
    
    # join the true cols without empty cells to one column and encode the true and pred dimension cols

    dataset['dimension_true'] = dataset[dimension_cols].apply(lambda row: ', '.join(row.dropna().astype(str)), axis=1)
    dataset['dimension_true_encoded'] = dataset['dimension_true'].apply(lambda text: encode_labels(text, label_space))
    dataset['dimension_pred_encoded'] = dataset[pred_dimension_col].apply(lambda text: encode_labels(text, label_space) if isinstance(text, str) else [0]*len(label_space))

    dimension_micro_f1 = f1_score(dataset['dimension_true_encoded'].tolist(), dataset['dimension_pred_encoded'].tolist(), average='micro', zero_division=0)
    dimension_macro_f1 = f1_score(dataset['dimension_true_encoded'].tolist(), dataset['dimension_pred_encoded'].tolist(), average='macro', zero_division=0)
    dimension_accuracy = accuracy_score(dataset['dimension_true_encoded'].tolist(), dataset['dimension_pred_encoded'].tolist())
    #full report as table
    dimension_full_report = classification_report(dataset['dimension_true_encoded'].tolist(), dataset['dimension_pred_encoded'].tolist(), target_names=label_space, zero_division=0, digits=3)

    # In addition, we check if at least one true dimension is included in the predicted dimensions (which may be a comma-separated string)
    def check_dimension_match(row):
        #true_dim = row[truth_dimension_col]
        true_dims = [row[col] for col in dimension_cols if pd.notnull(row[col])]
        pred_raw = row[pred_dimension_col]
        if all(pd.isnull(true_dims)) and pd.isnull(pred_raw):
            return True  # Both are NaN = match
        elif pd.isnull(pred_raw):
            return False  # No prediction = no match
        else:
            pred_dims = [dim.strip() for dim in pred_raw.split(',')]
            return any(true_dim in pred_dims for true_dim in true_dims)

    if not dataset.empty:
        print('\nFull per-label Dimension classification report:\n')
        print(dimension_full_report)
        print(f'MAP Dimension Accuracy:{dimension_accuracy:.3f}')
        dataset['dimension_match'] = dataset.apply(check_dimension_match, axis=1)
        dimension_accuracy_alternative = dataset['dimension_match'].mean()
        print(f'MAP Dimension Accuracy (both N/A = match, at least one match): {dimension_accuracy_alternative:.3f}')
    else:
        print('No valid rows for MAP Dimension evaluation.')

    # Return the evaluation results as a dictionary
    return {
        "model_id": "BoW",
        "system_idx": "NA",
        "user_idx": "NA",
        "dropped_explicit": 0,
        "dropped_implicit": 0,
        'explicit_accuracy': exp_accuracy,
        'explicit_precision_yes': exp_precision_yes,
        'explicit_recall_yes': exp_recall_yes,
        'explicit_f1_yes': exp_f1_yes,
        'explicit_precision_no': exp_precision_no,
        'explicit_recall_no': exp_recall_no,
        'explicit_f1_no': exp_f1_no,
        'implicit_accuracy': imp_accuracy,
        'implicit_precision_yes': imp_precision_yes,
        'implicit_recall_yes': imp_recall_yes,
        'implicit_f1_yes': imp_f1_yes,
        'implicit_precision_no': imp_precision_no,
        'implicit_recall_no': imp_recall_no,
        'implicit_f1_no': imp_f1_no,
        'dimension_percentage_false': dim_percentage,
        'dimension_micro_f1': dimension_micro_f1,
        'dimension_macro_f1': dimension_macro_f1,
        'dimension_accuracy': dimension_accuracy,
        'dimension_accuracy_alternative': dimension_accuracy_alternative,
        'dimension_full_report': dimension_full_report
    }

# Run the evaluation function on the validation dataset
evaluation_result = evaluate_Bow(validation_df)

if os.path.exists(f"GLLM/validation_summary_total.xlsx"):
    results_df = pd.read_excel(f"GLLM/validation_summary_total.xlsx")
    results_df = pd.concat([results_df, pd.DataFrame([evaluation_result])], ignore_index=True)
    results_df.to_excel(f"GLLM/validation_summary_total.xlsx", index=False)
else:
    results_df = pd.DataFrame([evaluation_result])
    results_df.to_excel(f"GLLM/validation_summary_total.xlsx", index=False)